> **Eleição presidencial de 2022.** Notebook novo, escrito e executado em outubro de 2026 com os dados do TSE de 2022. Ele substitui o `00_Geolocalizacao`, de 2022, que buscava o endereço de cada zona no Google Maps. Veja o [README](../README.md).

# Coordenadas das Zonas Eleitorais (Eleição de 2022)

Cada zona eleitoral vira um ponto nos mapas dos notebooks 03 e 04. O ponto sai dos dados abertos do TSE: o arquivo *Eleitorado por local de votação - 2022* traz a latitude e a longitude do local de votação de cada seção.

- **Zona com coordenadas:** o ponto é a mediana da latitude e da longitude das seções, ponderada pelo número de eleitores de cada uma. A mediana não se deixa levar por um local com coordenada errada, como a média faria.
- **Coordenada fora do estado:** uma seção cuja coordenada cai a mais de 20 km do próprio estado conta como seção sem coordenada. Em algumas zonas, o arquivo põe os locais de votação em outro estado.
- **Zona sem nenhuma seção com coordenada:** o ponto é a mediana, ponderada da mesma forma, dos centroides dos municípios das seções, tirados da malha do IBGE.
- **Zona do exterior (ZZ):** recebe o centroide de Brasília, onde fica o cartório dela. Os notebooks 03 e 04 deixam essa zona de fora.

In [1]:
import os
import zipfile
import unicodedata
import urllib.request

import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
PASTA_DO_PROJETO = '..'  # raiz do repositório

# Lê a Lista das Zonas Eleitorais

In [3]:
#Arquivos baixados do site do TSE em outubro de 2022
#https://www.tse.jus.br/eleitor/cartorios-e-zonas-eleitorais/pesquisa-a-zonas-eleitorais

path_zonas = f'{PASTA_DO_PROJETO}/dados_zonas_eleitorais'

def leitura_zonas(pasta, arq_csv):
    df = pd.read_csv(f'{pasta}/{arq_csv}', sep = ',', encoding = 'latin-1', dtype = str)
    return df.drop('sigla_uf', axis = 1).rename({'sigla_uf.1': 'sigla_uf'}, axis = 1)

df_zonas = pd.concat([leitura_zonas(path_zonas, arq) for arq in sorted(os.listdir(path_zonas))]).reset_index(drop = True)
df_zonas['numero_zona'] = df_zonas['numero_zona'].astype(int)
print(df_zonas.shape)

(2637, 6)


# Lê os Locais de Votação

In [4]:
#Eleitorado por local de votação - 2022 (TSE)
#https://dadosabertos.tse.jus.br/dataset/eleitorado-2022
url_locais = 'https://cdn.tse.jus.br/estatistica/sead/odsele/eleitorado_locais_votacao/eleitorado_local_votacao_2022.zip'
path_locais = f'{PASTA_DO_PROJETO}/dados_locais_votacao'
arq_locais = f'{path_locais}/eleitorado_local_votacao_2022.zip'
if not os.path.exists(arq_locais):
    os.makedirs(path_locais, exist_ok = True)
    urllib.request.urlretrieve(url_locais, arq_locais)

cols = ['DT_GERACAO', 'NR_TURNO', 'SG_UF', 'NM_MUNICIPIO', 'NR_ZONA', 'NR_SECAO', 'QT_ELEITOR_SECAO', 'NR_LATITUDE', 'NR_LONGITUDE']
with zipfile.ZipFile(arq_locais) as z:
    with z.open('eleitorado_local_votacao_2022.csv') as f:
        df_locais = pd.read_csv(f, sep = ';', encoding = 'latin-1', usecols = cols)
df_locais = df_locais[df_locais['NR_TURNO'] == 1].reset_index(drop = True)

print(f"Arquivo gerado pelo TSE em {df_locais['DT_GERACAO'].iloc[0]}")
print(df_locais.shape)

Arquivo gerado pelo TSE em 30/09/2024
(496856, 9)


In [5]:
#Seções sem coordenada vêm com -1
df_locais['COM_COORDENADA'] = (df_locais['NR_LATITUDE'] != -1) & (df_locais['NR_LONGITUDE'] != -1)

#Seções com coordenada fora do próprio estado, a mais de 20 km dele, também contam como sem coordenada:
#em algumas zonas, o arquivo põe os locais de votação em outro estado
path_malhas = f'{PASTA_DO_PROJETO}/dados_malhas_ibge'
gdf_uf = gpd.read_file(f'{path_malhas}/BR_UF_2021.zip')
uf_plana = gdf_uf.set_index('SIGLA')['geometry'].to_crs(5880)
mask = df_locais['COM_COORDENADA'] & (df_locais['SG_UF'] != 'ZZ')
gdf_secoes = gpd.GeoDataFrame(df_locais.loc[mask, ['SG_UF', 'NR_ZONA']], crs = gdf_uf.crs,
                              geometry = gpd.points_from_xy(df_locais.loc[mask, 'NR_LONGITUDE'], df_locais.loc[mask, 'NR_LATITUDE']))
gdf_secoes = gpd.sjoin(gdf_secoes, gdf_uf[['SIGLA', 'geometry']], how = 'left', predicate = 'within')
gdf_secoes = gdf_secoes[~gdf_secoes.index.duplicated()]
gdf_fora = gdf_secoes[gdf_secoes['SIGLA'] != gdf_secoes['SG_UF']].to_crs(5880)
gdf_fora['dist_estado_km'] = [uf_plana[uf].distance(p)/1000 for uf, p in zip(gdf_fora['SG_UF'], gdf_fora['geometry'])]
gdf_fora = gdf_fora[gdf_fora['dist_estado_km'] > 20]
df_locais.loc[gdf_fora.index, 'COM_COORDENADA'] = False
print(f'Seções descartadas por cair a mais de 20 km do próprio estado: {gdf_fora.shape[0]} (em {gdf_fora.groupby(["SG_UF", "NR_ZONA"]).ngroups} zonas)')

df_secoes = df_locais.groupby(['SG_UF', 'NR_ZONA']).agg(secoes = ('NR_SECAO', 'size'),
                                                       secoes_com_coordenada = ('COM_COORDENADA', 'sum')).reset_index()

print(f"Seções com coordenada: {df_locais['COM_COORDENADA'].sum()} de {df_locais.shape[0]} ({100*df_locais['COM_COORDENADA'].mean():.1f}%)")
print(f"Zonas sem nenhuma seção com coordenada: {(df_secoes['secoes_com_coordenada'] == 0).sum()} de {df_secoes.shape[0]}")

Seções descartadas por cair a mais de 20 km do próprio estado: 289 (em 31 zonas)
Seções com coordenada: 467067 de 496856 (94.0%)
Zonas sem nenhuma seção com coordenada: 87 de 2637


# Ponto de Cada Zona

In [6]:
def mediana_ponderada(valores, pesos):
    ordem = np.argsort(valores)
    acumulado = np.cumsum(pesos[ordem])
    return valores[ordem][np.searchsorted(acumulado, acumulado[-1]/2)]

def ponto_da_zona(x, col_lat, col_long):
    pesos = x['QT_ELEITOR_SECAO'].clip(lower = 1).values
    return pd.Series({'Latitude': mediana_ponderada(x[col_lat].values, pesos),
                      'Longitude': mediana_ponderada(x[col_long].values, pesos)})

cols = ['NR_LATITUDE', 'NR_LONGITUDE', 'QT_ELEITOR_SECAO']
df_coord = df_locais[df_locais['COM_COORDENADA']].groupby(['SG_UF', 'NR_ZONA'])[cols].apply(
    lambda x: ponto_da_zona(x, 'NR_LATITUDE', 'NR_LONGITUDE')).reset_index()
df_coord['origem'] = 'locais de votação'
print(df_coord.shape)

(2550, 5)


### Zonas sem Coordenada: Centroide dos Municípios

In [7]:
gdf_mun = gpd.read_file(f'{path_malhas}/BR_Municipios_2021.zip')
#Centroide calculado numa projeção plana (SIRGAS 2000 / Brazil Polyconic) e convertido de volta para latitude e longitude
centroides = gdf_mun['geometry'].to_crs(5880).centroid.to_crs(gdf_mun.crs)
#Os nomes são comparados sem acento: o TSE escreve, por exemplo, CAEM, e o IBGE, Caém
def sem_acento(nome):
    return ''.join(c for c in unicodedata.normalize('NFKD', nome.upper()) if not unicodedata.combining(c))

df_mun = pd.DataFrame({'SG_UF': gdf_mun['SIGLA'].values, 'NM_MUNICIPIO': gdf_mun['NM_MUN'].str.upper().values,
                       'CHAVE_MUN': gdf_mun['NM_MUN'].apply(sem_acento).values,
                       'LAT_MUN': centroides.y.values, 'LONG_MUN': centroides.x.values})
assert not df_mun.duplicated(['SG_UF', 'CHAVE_MUN']).any()

zonas_sem = df_secoes.loc[(df_secoes['secoes_com_coordenada'] == 0) & (df_secoes['SG_UF'] != 'ZZ'), ['SG_UF', 'NR_ZONA']]
df_sem = pd.merge(df_locais, zonas_sem, on = ['SG_UF', 'NR_ZONA'])
df_sem['CHAVE_MUN'] = df_sem['NM_MUNICIPIO'].apply(sem_acento)
df_sem = pd.merge(df_sem, df_mun.drop('NM_MUNICIPIO', axis = 1), how = 'left', on = ['SG_UF', 'CHAVE_MUN'])
print('Seções sem município correspondente na malha do IBGE:', df_sem['LAT_MUN'].isna().sum())

cols = ['LAT_MUN', 'LONG_MUN', 'QT_ELEITOR_SECAO']
df_coord_mun = df_sem.dropna(subset = ['LAT_MUN']).groupby(['SG_UF', 'NR_ZONA'])[cols].apply(
    lambda x: ponto_da_zona(x, 'LAT_MUN', 'LONG_MUN')).reset_index()
df_coord_mun['origem'] = 'centroide dos municípios'
print(df_coord_mun.shape)

Seções sem município correspondente na malha do IBGE: 0
(86, 5)


In [8]:
#Zona do exterior: centroide de Brasília, onde fica o cartório
brasilia = df_mun[(df_mun['SG_UF'] == 'DF') & (df_mun['NM_MUNICIPIO'] == 'BRASÍLIA')].iloc[0]
df_coord_zz = pd.DataFrame([{'SG_UF': 'ZZ', 'NR_ZONA': 1, 'Latitude': brasilia['LAT_MUN'], 'Longitude': brasilia['LONG_MUN'],
                             'origem': 'cartório em Brasília'}])

# Junta e Confere

In [9]:
renomear = {'SG_UF': 'sigla_uf', 'NR_ZONA': 'numero_zona'}
df_pontos = pd.concat([df_coord, df_coord_mun, df_coord_zz]).rename(renomear, axis = 1)
df_pontos = pd.merge(df_pontos, df_secoes.rename(renomear, axis = 1), how = 'left', on = ['sigla_uf', 'numero_zona'])

cols_zona = ['numero_zona', 'sigla_uf', 'nome_municipio', 'bairro']
df_saida = pd.merge(df_zonas[cols_zona], df_pontos, how = 'left', on = ['sigla_uf', 'numero_zona'])
df_saida[['Latitude', 'Longitude']] = df_saida[['Latitude', 'Longitude']].round(6)

assert df_saida.shape[0] == df_zonas.shape[0] and df_saida['Latitude'].notna().all()
df_saida['origem'].value_counts()

origem
locais de votação           2550
centroide dos municípios      86
cartório em Brasília           1
Name: count, dtype: int64

In [10]:
#Confere se o ponto de cada zona cai dentro do próprio estado
gdf_pontos = gpd.GeoDataFrame(df_saida[df_saida['sigla_uf'] != 'ZZ'], crs = gdf_uf.crs,
                              geometry = gpd.points_from_xy(df_saida.loc[df_saida['sigla_uf'] != 'ZZ', 'Longitude'],
                                                            df_saida.loc[df_saida['sigla_uf'] != 'ZZ', 'Latitude']))
gdf_pontos = gpd.sjoin(gdf_pontos, gdf_uf[['SIGLA', 'geometry']], how = 'left', predicate = 'within')
fora = gdf_pontos[gdf_pontos['SIGLA'] != gdf_pontos['sigla_uf']].copy()
fora['dist_estado_km'] = [uf_plana[uf].distance(p)/1000 for uf, p in zip(fora['sigla_uf'], fora['geometry'].to_crs(5880))]
print(f'Zonas com o ponto fora do próprio estado: {fora.shape[0]} de {gdf_pontos.shape[0]}')
fora[cols_zona + ['origem', 'SIGLA', 'dist_estado_km']].round(1)

Zonas com o ponto fora do próprio estado: 4 de 2636


,numero_zona,sigla_uf,nome_municipio,bairro,origem,SIGLA,dist_estado_km
1030,33,MS,MUNDO NOVO,CENTRO,locais de votação,NaN,5.3
1738,116,RJ,ANGRA DOS REIS,CENTRO,locais de votação,NaN,0.1
1759,147,RJ,ANGRA DOS REIS,SAO BENTO,locais de votação,NaN,0.6
1777,172,RJ,ARMAÇÃO DOS BÚZIOS,CENTRO,locais de votação,NaN,0.2


# Salva

In [11]:
path_output = f'{PASTA_DO_PROJETO}/dados_coordenadas_zonas'
os.makedirs(path_output, exist_ok = True)
df_saida.to_csv(f'{path_output}/coordenadas_zonas_eleitorais_2022.csv', index = False)
df_saida.sample(n = 5, random_state = 1)

,numero_zona,sigla_uf,nome_municipio,bairro,Latitude,Longitude,origem,secoes,secoes_com_coordenada
1551,83,PR,SANTO ANTÔNIO DO SUDOESTE,CENTRO,-26.052773,-53.721713,locais de votação,85,80
785,106,MG,ENTRE RIOS DE MINAS,JARDIM PRIMAVERA,-20.661885,-44.066172,locais de votação,85,83
1104,61,MT,COMODORO,CENTRO,-13.654734,-59.782999,locais de votação,86,73
620,31,MA,ICATU,CENTRO - ZONA URBANA,-2.775562,-44.052413,locais de votação,124,124
1993,77,RS,OSÓRIO,CENTRO,-29.887541,-50.254840,locais de votação,167,167
